# 10 - Walk-Forward Tuning e Random Forest

**Regra dura: o conjunto de TESTE nao e carregado. Nenhuma mencao a ele. Sem tratamento
de desbalanceamento.**

Concatena train.parquet + validation.parquet (2007-2014) para poder redefinir janelas
temporais expansivas a partir de issue_d. O teste (2015) continua totalmente fora - nao
esta em nenhum dos dois arquivos carregados aqui.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

%matplotlib inline

PROCESSED_DIR = Path('..') / 'data' / 'processed'
FAMILY_COLOR = {'M0b': '#888888', 'M1': '#0072B2', 'XGB1': '#D55E00', 'RF': '#CC79A7',
                'XGB_tuned': '#009E73', 'XGB_walkforward': '#E69F00'}

train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
validation = pd.read_parquet(PROCESSED_DIR / 'validation.parquet')

full_data = pd.concat([train, validation], ignore_index=True).sort_values('issue_d').reset_index(drop=True)
print(f'train: {train.shape} | validation: {validation.shape} | full_data (concatenado): {full_data.shape}')
print(f'Intervalo de issue_d em full_data: {full_data["issue_d"].min()} a {full_data["issue_d"].max()}')


train: (172988, 89) | validation: (162570, 89) | full_data (concatenado): (335558, 89)
Intervalo de issue_d em full_data: 2007-06-01 00:00:00 a 2014-12-01 00:00:00


### Reconstrucao de FEATURE_SET, financeiro, prepare_X (identico aos notebooks 06-09)

In [2]:
EVAL_ONLY = ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
PROVISIONAL_EXCLUDE = ['int_rate', 'grade', 'sub_grade']

family_C_features = ['funded_amnt', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
    'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med',
    'mths_since_last_major_derog', 'application_type', 'acc_now_delinq', 'tot_coll_amt',
    'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
    'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
    'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
    'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts',
    'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
    'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
    'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
    'total_il_high_credit_limit', 'emp_length_anos']
assert len(family_C_features) == 65

engineered_flags = ['era_pre_2012',
                     'mths_since_last_delinq_missing', 'mths_since_last_record_missing',
                     'mths_since_recent_bc_dlq_missing', 'mths_since_recent_revol_delinq_missing',
                     'mths_since_last_major_derog_missing', 'emp_length_missing',
                     'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'sparse_bureau_missing']
assert len(engineered_flags) == 10

new_features = ['installment_to_income', 'loan_to_income', 'credit_history_months',
                 'revol_bal_to_income', 'open_acc_ratio']
assert len(new_features) == 5

redundant_cols = {'fico_range_high': 'redundancia (r=1.0 com fico_range_low)'}
FEATURE_SET = [c for c in family_C_features if c not in redundant_cols] + engineered_flags + new_features
assert len(FEATURE_SET) == 79

CATEGORICAL_COLS = ['home_ownership', 'purpose', 'verification_status', 'initial_list_status', 'application_type']
REFERENCE_DATE = pd.Timestamp('2000-01-01')


def compute_financials(df):
    interest = (df['installment'] * df['term']) - df['loan_amnt']
    loss_raw = df['loan_amnt'] - df['total_rec_prncp']
    return interest, loss_raw.clip(lower=0)


def prepare_X(df, feature_cols, categorical_cols):
    X = df[feature_cols].copy()
    for c in ['issue_d', 'earliest_cr_line']:
        if c in X.columns:
            X[c] = (X[c] - REFERENCE_DATE).dt.days
    cat_present = [c for c in categorical_cols if c in X.columns]
    X = pd.get_dummies(X, columns=cat_present, drop_first=True)
    return X


def profit_at_threshold(y_true, y_prob, threshold, interest, loss):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    interest = np.asarray(interest)
    loss = np.asarray(loss)
    aprovados = y_prob < threshold
    return interest[aprovados & (y_true == 0)].sum() - loss[aprovados & (y_true == 1)].sum()


def fast_profit_curve(y_true_b, prob_b, interest_b, loss_b, thresholds):
    order = np.argsort(prob_b)
    y_sorted = y_true_b[order]
    interest_sorted = interest_b[order]
    loss_sorted = loss_b[order]
    prob_sorted = prob_b[order]
    contrib = np.where(y_sorted == 0, interest_sorted, -loss_sorted)
    cumsum = np.cumsum(contrib)
    idx_cut = np.searchsorted(prob_sorted, thresholds, side='left')
    return np.where(idx_cut > 0, cumsum[np.clip(idx_cut - 1, 0, None)], 0.0)


def decompose(y_prob, threshold, y_true, interest, loss, loan_amnt):
    rejected = y_prob >= threshold
    rejected_co = rejected & (y_true == 1)
    rejected_fp = rejected & (y_true == 0)
    avoided_loss = loss[rejected_co].sum()
    lost_interest = interest[rejected_fp].sum()
    return {'threshold': threshold, 'n_rejeitados': int(rejected.sum()),
            'valor_total_rejeitado': loan_amnt[rejected].sum(),
            'n_rejeitados_charged_off': int(rejected_co.sum()), 'n_rejeitados_fully_paid': int(rejected_fp.sum()),
            'perda_evitada_$': avoided_loss, 'juros_perdidos_$': lost_interest,
            'ganho_liquido_$': avoided_loss - lost_interest}


threshold_grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
print('Infraestrutura recriada.')


Infraestrutura recriada.


## Secao 1 - Definicao das janelas walk-forward

- Janela 1 (W1): treino issue_d <= 2011-12-31, validacao 2012.
- Janela 2 (W2): treino <= 2012-12-31, validacao 2013.
- Janela 3 (W3): treino <= 2013-12-31, validacao 2014 (= o split original train/validation).

As linhas de 2012 e 2013 usadas como validacao em W1/W2 vem de dentro de train.parquet
(que cobre 2007-2013) - por isso a concatenacao no inicio era necessaria: sem ela, essas
safras nao estariam disponiveis como alvo de avaliacao isolada.

In [3]:
window_defs = [
    {'name': 'W1', 'train_end': pd.Timestamp('2011-12-31'), 'val_start': pd.Timestamp('2012-01-01'), 'val_end': pd.Timestamp('2012-12-31')},
    {'name': 'W2', 'train_end': pd.Timestamp('2012-12-31'), 'val_start': pd.Timestamp('2013-01-01'), 'val_end': pd.Timestamp('2013-12-31')},
    {'name': 'W3', 'train_end': pd.Timestamp('2013-12-31'), 'val_start': pd.Timestamp('2014-01-01'), 'val_end': pd.Timestamp('2014-12-31')},
]

windows = {}
window_summary_rows = []
for wd in window_defs:
    train_w = full_data.loc[full_data['issue_d'] <= wd['train_end']].copy()
    val_w = full_data.loc[(full_data['issue_d'] >= wd['val_start']) & (full_data['issue_d'] <= wd['val_end'])].copy()

    X_train_w = prepare_X(train_w, FEATURE_SET, CATEGORICAL_COLS)
    X_val_w = prepare_X(val_w, FEATURE_SET, CATEGORICAL_COLS)
    X_val_w = X_val_w.reindex(columns=X_train_w.columns, fill_value=0)

    interest_w, loss_w = compute_financials(val_w)

    windows[wd['name']] = {
        'X_train': X_train_w, 'y_train': train_w['target'].values,
        'X_val': X_val_w, 'y_val': val_w['target'].values,
        'interest': interest_w.values, 'loss': loss_w.values,
        'loan_amnt': val_w['loan_amnt'].values,
    }

    window_summary_rows.append({
        'janela': wd['name'], 'N_treino': len(train_w), 'N_validacao': len(val_w),
        'default_%_treino': round(train_w['target'].mean() * 100, 4),
        'default_%_validacao': round(val_w['target'].mean() * 100, 4),
        '%_era_pre_2012_no_treino': round(train_w['era_pre_2012'].mean() * 100, 4),
    })

pd.DataFrame(window_summary_rows).set_index('janela')


,N_treino,N_validacao,default_%_treino,default_%_validacao,%_era_pre_2012_no_treino
janela,,,,,
W1,29096,43470,11.0909,13.5795,100.0000
W2,72566,100422,12.5816,12.3260,71.1711
W3,172988,162570,12.4332,13.7264,29.8553


## Secao 2 - Random Forest default (a comparacao que faltava)

Treinado em train.parquet (W3), avaliado em validation.parquet (W3) - mesma janela usada
por M1/XGB1/XGB_tuned. Sem tuning - referencia de familia.

In [4]:
y_val_w3 = windows['W3']['y_val']
interest_w3 = windows['W3']['interest']
loss_w3 = windows['W3']['loss']
loan_amnt_w3 = windows['W3']['loan_amnt']

profit_m0b = profit_at_threshold(y_val_w3, np.zeros(len(y_val_w3)), 1.01, interest_w3, loss_w3)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(windows['W3']['X_train'], windows['W3']['y_train'])
y_prob_val_rf = rf.predict_proba(windows['W3']['X_val'])[:, 1]

auc_roc_rf = roc_auc_score(y_val_w3, y_prob_val_rf)
auc_pr_rf = average_precision_score(y_val_w3, y_prob_val_rf)
brier_rf = brier_score_loss(y_val_w3, y_prob_val_rf)

profits_rf = fast_profit_curve(y_val_w3, y_prob_val_rf, interest_w3, loss_w3, threshold_grid)
best_i_rf = int(np.argmax(profits_rf))
t_rf = threshold_grid[best_i_rf]
profit_rf = profits_rf[best_i_rf]
aprov_rf = y_prob_val_rf < t_rf

within_01pct_rf = threshold_grid[profits_rf >= profits_rf.max() * 0.999]

print(f'AUC-ROC: {auc_roc_rf:.4f} | AUC-PR: {auc_pr_rf:.4f} | Brier: {brier_rf:.4f}')
print(f'Threshold otimo: {t_rf} | Lucro otimo: $ {profit_rf:,.2f} | Lucro vs M0b: $ {profit_rf - profit_m0b:,.2f}')
print(f'% aprovados: {aprov_rf.mean() * 100:.2f}% | default entre aprovados: {y_val_w3[aprov_rf].mean() * 100:.4f}%')
print(f'Largura do plato a 0.1%: {len(within_01pct_rf)} thresholds [{within_01pct_rf.min():.2f}, {within_01pct_rf.max():.2f}]')


AUC-ROC: 0.6590 | AUC-PR: 0.2214 | Brier: 0.1148
Threshold otimo: 0.56 | Lucro otimo: $ 189,747,912.27 | Lucro vs M0b: $ 0.00
% aprovados: 100.00% | default entre aprovados: 13.7264%
Largura do plato a 0.1%: 65 thresholds [0.35, 0.99]


## Secao 3 - Re-tuning do XGBoost com validacao walk-forward

Mesmo espaco de busca e mesma seed do notebook 09 - as 30 combinacoes amostradas sao
identicas, para comparar selecao walk-forward vs. janela-unica de forma controlada.

In [5]:
param_distributions = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 400, 600],
    'min_child_weight': [1, 5, 10, 20],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

sampler = ParameterSampler(param_distributions, n_iter=30, random_state=42)
param_list = list(sampler)
print(f'{len(param_list)} combinacoes amostradas (identicas ao notebook 09).')


30 combinacoes amostradas (identicas ao notebook 09).


In [6]:
wf_rows = []
for i, params in enumerate(param_list):
    profits_by_window = {}
    for wname in ['W1', 'W2', 'W3']:
        w = windows[wname]
        model = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        model.fit(w['X_train'], w['y_train'])
        y_prob = model.predict_proba(w['X_val'])[:, 1]
        profits = fast_profit_curve(w['y_val'], y_prob, w['interest'], w['loss'], threshold_grid)
        profits_by_window[wname] = profits.max()

    mean_profit = np.mean(list(profits_by_window.values()))
    row = dict(params)
    row.update({'combinacao': i, 'lucro_W1': profits_by_window['W1'], 'lucro_W2': profits_by_window['W2'],
                'lucro_W3': profits_by_window['W3'], 'lucro_medio': mean_profit})
    wf_rows.append(row)
    print(f'[{i + 1:02d}/30] W1=$ {profits_by_window["W1"]:,.0f} | W2=$ {profits_by_window["W2"]:,.0f} | '
          f'W3=$ {profits_by_window["W3"]:,.0f} | media=$ {mean_profit:,.0f}')

wf_df = pd.DataFrame(wf_rows).sort_values('lucro_medio', ascending=False).reset_index(drop=True)
print()
print('Ranking por lucro medio nas 3 janelas:')


[01/30] W1=$ 52,161,174 | W2=$ 148,025,078 | W3=$ 191,258,601 | media=$ 130,481,618


[02/30] W1=$ 52,150,159 | W2=$ 147,975,015 | W3=$ 190,119,953 | media=$ 130,081,709


[03/30] W1=$ 52,249,388 | W2=$ 147,985,261 | W3=$ 191,511,221 | media=$ 130,581,957


[04/30] W1=$ 52,366,827 | W2=$ 148,119,695 | W3=$ 191,264,438 | media=$ 130,583,653


[05/30] W1=$ 52,135,581 | W2=$ 148,098,493 | W3=$ 190,805,872 | media=$ 130,346,649


...

[26/30] W1=$ 52,466,502 | W2=$ 148,222,203 | W3=$ 190,992,237 | media=$ 130,560,314


[27/30] W1=$ 52,394,493 | W2=$ 147,991,277 | W3=$ 190,570,181 | media=$ 130,318,651


[28/30] W1=$ 52,094,924 | W2=$ 148,011,988 | W3=$ 190,507,351 | media=$ 130,204,754


[29/30] W1=$ 52,136,014 | W2=$ 147,994,652 | W3=$ 189,990,759 | media=$ 130,040,475


[30/30] W1=$ 52,123,034 | W2=$ 148,011,251 | W3=$ 190,317,783 | media=$ 130,150,689

Ranking por lucro medio nas 3 janelas:


In [7]:
wf_df


,subsample,n_estimators,min_child_weight,max_depth,learning_rate,colsample_bytree,combinacao,lucro_W1,lucro_W2,lucro_W3,lucro_medio
0,0.8,600,10,8,0.03,0.6,9,5.225161e+07,1.480654e+08,1.917577e+08,1.306916e+08
1,0.6,400,20,3,0.03,0.8,12,5.244190e+07,1.482651e+08,1.911488e+08,1.306186e+08
2,0.8,600,20,4,0.01,0.8,3,5.236683e+07,1.481197e+08,1.912644e+08,1.305837e+08
3,1.0,400,20,5,0.10,0.6,2,5.224939e+07,1.479853e+08,1.915112e+08,1.305820e+08
4,1.0,400,5,3,0.20,1.0,19,5.214819e+07,1.480333e+08,1.915524e+08,1.305780e+08
5,0.6,600,5,8,0.03,1.0,24,5.215982e+07,1.480842e+08,1.914400e+08,1.305614e+08
6,0.6,600,10,3,0.01,1.0,25,5.246650e+07,1.482222e+08,1.909922e+08,1.305603e+08
7,0.6,200,10,4,0.03,0.8,22,5.243534e+07,1.480944e+08,1.911098e+08,1.305465e+08
8,0.6,400,1,8,0.03,0.8,7,5.212779e+07,1.482713e+08,1.912160e+08,1.305384e+08
9,0.8,600,10,5,0.01,0.6,17,5.233602e+07,1.481735e+08,1.910651e+08,1.305249e+08


## Secao 4 - Diagnostico de estacionariedade

In [8]:
int_params = {'max_depth', 'n_estimators', 'min_child_weight'}

def row_to_params(row):
    return {k: (int(row[k]) if k in int_params else float(row[k])) for k in param_distributions}


winner_mean = wf_df.iloc[0]
print('1. Combinacao vencedora por LUCRO MEDIO (as 3 janelas):')
print(row_to_params(winner_mean))
print()

winners_by_window = {}
for wname, col in [('W1', 'lucro_W1'), ('W2', 'lucro_W2'), ('W3', 'lucro_W3')]:
    best_row = wf_df.sort_values(col, ascending=False).iloc[0]
    winners_by_window[wname] = best_row
    print(f'Combinacao vencedora isolada de {wname}: {row_to_params(best_row)} -> $ {best_row[col]:,.2f}')

print()
same_winner = all((winners_by_window[w][list(param_distributions)] == winner_mean[list(param_distributions)]).all()
                   for w in ['W1', 'W2', 'W3'])
print(f'A vencedora por media e a mesma que venceria em TODAS as janelas isoladas? {same_winner}')


1. Combinacao vencedora por LUCRO MEDIO (as 3 janelas):
{'max_depth': 8, 'learning_rate': 0.03, 'n_estimators': 600, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.6}

Combinacao vencedora isolada de W1: {'max_depth': 3, 'learning_rate': 0.01, 'n_estimators': 600, 'min_child_weight': 10, 'subsample': 0.6, 'colsample_bytree': 1.0} -> $ 52,466,502.43
Combinacao vencedora isolada de W2: {'max_depth': 4, 'learning_rate': 0.03, 'n_estimators': 400, 'min_child_weight': 20, 'subsample': 0.6, 'colsample_bytree': 0.6} -> $ 148,299,410.48
Combinacao vencedora isolada de W3: {'max_depth': 8, 'learning_rate': 0.03, 'n_estimators': 600, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.6} -> $ 191,757,692.42

A vencedora por media e a mesma que venceria em TODAS as janelas isoladas? False


In [9]:
print('2. Hiperparametros (max_depth, learning_rate, n_estimators) do vencedor de cada janela:')
for wname in ['W1', 'W2', 'W3']:
    r = winners_by_window[wname]
    print(f'  {wname}: max_depth={int(r["max_depth"])} | learning_rate={r["learning_rate"]} | n_estimators={int(r["n_estimators"])}')
print(f'  Vencedor por media: max_depth={int(winner_mean["max_depth"])} | learning_rate={winner_mean["learning_rate"]} | '
      f'n_estimators={int(winner_mean["n_estimators"])}')


2. Hiperparametros (max_depth, learning_rate, n_estimators) do vencedor de cada janela:
  W1: max_depth=3 | learning_rate=0.01 | n_estimators=600
  W2: max_depth=4 | learning_rate=0.03 | n_estimators=400
  W3: max_depth=8 | learning_rate=0.03 | n_estimators=600
  Vencedor por media: max_depth=8 | learning_rate=0.03 | n_estimators=600


In [10]:
notebook09_params = {'max_depth': 6, 'learning_rate': 0.2, 'n_estimators': 100, 'min_child_weight': 10,
                      'subsample': 1.0, 'colsample_bytree': 0.6}

mask = pd.Series(True, index=wf_df.index)
for k, v in notebook09_params.items():
    mask &= (wf_df[k] == v)

match = wf_df.loc[mask]
if len(match) == 1:
    rank_position = match.index[0] + 1
    print(f'3. A combinacao vencedora do notebook 09 (janela unica 2014) e a linha #{rank_position} de 30 '
          f'no ranking walk-forward por lucro medio.')
    print(f'   Lucro medio dela nas 3 janelas: $ {match.iloc[0]["lucro_medio"]:,.2f}')
    print(f'   Lucro medio do vencedor walk-forward: $ {winner_mean["lucro_medio"]:,.2f}')
else:
    print(f'3. ATENCAO: encontrei {len(match)} linhas correspondentes aos hiperparametros do notebook 09 (esperado 1).')


3. A combinacao vencedora do notebook 09 (janela unica 2014) e a linha #19 de 30 no ranking walk-forward por lucro medio.
   Lucro medio dela nas 3 janelas: $ 130,346,648.55
   Lucro medio do vencedor walk-forward: $ 130,691,564.38


## Secao 5 - O modelo walk-forward final

Treinado com a combinacao vencedora por lucro medio, em train.parquet (2007-2013, = W3
treino) e avaliado em validation.parquet (2014, = W3 validacao) - a mesma janela usada
por todos os outros modelos, para comparabilidade direta.

In [11]:
wf_best_params = row_to_params(winner_mean)
xgb_wf = XGBClassifier(**wf_best_params, random_state=42, eval_metric='logloss')
xgb_wf.fit(windows['W3']['X_train'], windows['W3']['y_train'])
y_prob_val_wf = xgb_wf.predict_proba(windows['W3']['X_val'])[:, 1]

auc_roc_wf = roc_auc_score(y_val_w3, y_prob_val_wf)
auc_pr_wf = average_precision_score(y_val_w3, y_prob_val_wf)
brier_wf = brier_score_loss(y_val_w3, y_prob_val_wf)

profits_wf = fast_profit_curve(y_val_w3, y_prob_val_wf, interest_w3, loss_w3, threshold_grid)
best_i_wf = int(np.argmax(profits_wf))
t_wf = threshold_grid[best_i_wf]
profit_wf = profits_wf[best_i_wf]
aprov_wf = y_prob_val_wf < t_wf
within_01pct_wf = threshold_grid[profits_wf >= profits_wf.max() * 0.999]

print(f'Hiperparametros: {wf_best_params}')
print(f'AUC-ROC: {auc_roc_wf:.4f} | AUC-PR: {auc_pr_wf:.4f} | Brier: {brier_wf:.4f}')
print(f'Threshold otimo: {t_wf} | Lucro otimo: $ {profit_wf:,.2f} | Lucro vs M0b: $ {profit_wf - profit_m0b:,.2f}')
print(f'% aprovados: {aprov_wf.mean() * 100:.2f}% | default entre aprovados: {y_val_w3[aprov_wf].mean() * 100:.4f}%')
print(f'Largura do plato a 0.1%: {len(within_01pct_wf)} thresholds [{within_01pct_wf.min():.2f}, {within_01pct_wf.max():.2f}]')


Hiperparametros: {'max_depth': 8, 'learning_rate': 0.03, 'n_estimators': 600, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.6}
AUC-ROC: 0.6815 | AUC-PR: 0.2432 | Brier: 0.1128
Threshold otimo: 0.33 | Lucro otimo: $ 191,757,692.42 | Lucro vs M0b: $ 2,009,780.15
% aprovados: 97.27% | default entre aprovados: 13.1288%
Largura do plato a 0.1%: 2 thresholds [0.32, 0.33]


### Reconstruindo M1, XGB1 e XGB_tuned (janela unica, notebook 09) para a tabela comparativa

In [12]:
X_train_w3 = windows['W3']['X_train']
X_val_w3 = windows['W3']['X_val']
y_train_w3 = windows['W3']['y_train']

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_w3), columns=X_train_w3.columns, index=X_train_w3.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_w3), columns=X_val_w3.columns, index=X_val_w3.index)
m1_model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=2000, random_state=42)
m1_model.fit(X_train_scaled, y_train_w3)
y_prob_val_m1 = m1_model.predict_proba(X_val_scaled)[:, 1]
auc_m1 = roc_auc_score(y_val_w3, y_prob_val_m1)
profits_m1 = fast_profit_curve(y_val_w3, y_prob_val_m1, interest_w3, loss_w3, threshold_grid)
profit_m1 = profits_m1.max()

xgb1 = XGBClassifier(random_state=42, eval_metric='logloss')
xgb1.fit(X_train_w3, y_train_w3)
y_prob_val_xgb1 = xgb1.predict_proba(X_val_w3)[:, 1]
auc_xgb1 = roc_auc_score(y_val_w3, y_prob_val_xgb1)
profits_xgb1 = fast_profit_curve(y_val_w3, y_prob_val_xgb1, interest_w3, loss_w3, threshold_grid)
profit_xgb1 = profits_xgb1.max()

notebook09_best = {'max_depth': 6, 'learning_rate': 0.2, 'n_estimators': 100, 'min_child_weight': 10,
                    'subsample': 1.0, 'colsample_bytree': 0.6}
xgb_tuned = XGBClassifier(**notebook09_best, random_state=42, eval_metric='logloss')
xgb_tuned.fit(X_train_w3, y_train_w3)
y_prob_val_tuned = xgb_tuned.predict_proba(X_val_w3)[:, 1]
auc_tuned = roc_auc_score(y_val_w3, y_prob_val_tuned)
profits_tuned = fast_profit_curve(y_val_w3, y_prob_val_tuned, interest_w3, loss_w3, threshold_grid)
profit_tuned = profits_tuned.max()

print('M1, XGB1 e XGB_tuned (janela unica) reconstruidos.')


C:\Users\Avell\Documents\Projetos\credit-default-prediction-lendingclub\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


M1, XGB1 e XGB_tuned (janela unica) reconstruidos.


In [13]:
comparison_rows = [
    {'modelo': 'M0b (aprova todos)', 'auc_roc': np.nan, 'lucro_otimo': profit_m0b},
    {'modelo': 'M1 (LogReg)', 'auc_roc': round(auc_m1, 4), 'lucro_otimo': profit_m1},
    {'modelo': 'XGB1 (default)', 'auc_roc': round(auc_xgb1, 4), 'lucro_otimo': profit_xgb1},
    {'modelo': 'RF_default', 'auc_roc': round(auc_roc_rf, 4), 'lucro_otimo': profit_rf},
    {'modelo': 'XGB_tuned (janela unica, nb09)', 'auc_roc': round(auc_tuned, 4), 'lucro_otimo': profit_tuned},
    {'modelo': 'XGB_walkforward (este notebook)', 'auc_roc': round(auc_roc_wf, 4), 'lucro_otimo': profit_wf},
]
comparison_df = pd.DataFrame(comparison_rows).set_index('modelo')
comparison_df['lucro_vs_M0b'] = comparison_df['lucro_otimo'] - profit_m0b
comparison_df


,auc_roc,lucro_otimo,lucro_vs_M0b
modelo,,,
M0b (aprova todos),NaN,1.897479e+08,0.000000e+00
M1 (LogReg),0.6765,1.906221e+08,8.742220e+05
XGB1 (default),0.6606,1.898301e+08,8.221303e+04
RF_default,0.6590,1.897479e+08,6.675720e-06
"XGB_tuned (janela unica, nb09)",0.6764,1.908059e+08,1.057959e+06
XGB_walkforward (este notebook),0.6815,1.917577e+08,2.009780e+06


## Secao 6 - Bootstrap (1.000 reamostras da validacao 2014, seed=42, threshold fixo)

In [14]:
N_BOOT = 1000
SEED = 42
rng = np.random.default_rng(SEED)
n_val = len(y_val_w3)

y_prob_models = {'XGB_walkforward': y_prob_val_wf, 'M1': y_prob_val_m1, 'XGB_tuned': y_prob_val_tuned}
thresholds_fixed = {'XGB_walkforward': t_wf, 'M1': 0.38, 'XGB_tuned': float(threshold_grid[int(np.argmax(profits_tuned))])}

m0b_profit_boot = np.zeros(N_BOOT)
model_profit_boot = {name: np.zeros(N_BOOT) for name in y_prob_models}

for b in range(N_BOOT):
    idx = rng.integers(0, n_val, size=n_val)
    yb = y_val_w3[idx]
    ib = interest_w3[idx]
    lb = loss_w3[idx]
    m0b_profit_boot[b] = ib[yb == 0].sum() - lb[yb == 1].sum()
    for name, y_prob in y_prob_models.items():
        pb = y_prob[idx]
        approved = pb < thresholds_fixed[name]
        model_profit_boot[name][b] = ib[approved & (yb == 0)].sum() - lb[approved & (yb == 1)].sum()

diff_boot = {name: model_profit_boot[name] - m0b_profit_boot for name in y_prob_models}
print(f'Bootstrap concluido. N_BOOT={N_BOOT} seed={SEED}')


Bootstrap concluido. N_BOOT=1000 seed=42


In [15]:
d = diff_boot['XGB_walkforward']
ci_low, ci_high = np.percentile(d, [2.5, 97.5])
print('XGB_walkforward vs M0b:')
print(f'  media: $ {d.mean():,.2f} | IC 95%: [$ {ci_low:,.2f}, $ {ci_high:,.2f}] | cruza zero? {ci_low < 0 < ci_high}')
print(f'  % reamostras positivas: {(d > 0).mean() * 100:.2f}%')


XGB_walkforward vs M0b:
  media: $ 2,007,604.66 | IC 95%: [$ 1,120,035.82, $ 2,889,183.87] | cruza zero? False
  % reamostras positivas: 100.00%


In [16]:
diff_wf_m1 = model_profit_boot['XGB_walkforward'] - model_profit_boot['M1']
ci_low, ci_high = np.percentile(diff_wf_m1, [2.5, 97.5])
print('XGB_walkforward vs M1 (pareado):')
print(f'  media: $ {diff_wf_m1.mean():,.2f} | IC 95%: [$ {ci_low:,.2f}, $ {ci_high:,.2f}] | cruza zero? {ci_low < 0 < ci_high}')
print(f'  % de reamostras em que XGB_walkforward venceu: {(diff_wf_m1 > 0).mean() * 100:.2f}%')


XGB_walkforward vs M1 (pareado):
  media: $ 1,125,445.42 | IC 95%: [$ 278,777.62, $ 1,982,918.44] | cruza zero? False
  % de reamostras em que XGB_walkforward venceu: 99.40%


In [17]:
diff_wf_tuned = model_profit_boot['XGB_walkforward'] - model_profit_boot['XGB_tuned']
ci_low, ci_high = np.percentile(diff_wf_tuned, [2.5, 97.5])
print('XGB_walkforward vs XGB_tuned (pareado - o walk-forward mudou o resultado de forma detectavel?):')
print(f'  media: $ {diff_wf_tuned.mean():,.2f} | IC 95%: [$ {ci_low:,.2f}, $ {ci_high:,.2f}] | cruza zero? {ci_low < 0 < ci_high}')
print(f'  % de reamostras em que XGB_walkforward venceu XGB_tuned: {(diff_wf_tuned > 0).mean() * 100:.2f}%')


XGB_walkforward vs XGB_tuned (pareado - o walk-forward mudou o resultado de forma detectavel?):
  media: $ 956,432.38 | IC 95%: [$ 19,538.79, $ 1,869,180.12] | cruza zero? False
  % de reamostras em que XGB_walkforward venceu XGB_tuned: 97.80%


## Secao 7 - Nota de leitura

**(a) O walk-forward escolheu uma combinacao diferente da janela unica (notebook 09) -
mas em boa parte por um motivo que nao e conceitual.**

A combinacao vencedora por lucro medio (max_depth=8, learning_rate=0.03, n_estimators=600,
min_child_weight=10, subsample=0.8, colsample_bytree=0.6) e EXATAMENTE a mesma que venceria
isoladamente em W3 (2014) - a janela maior e mais recente, que domina a media simplesmente
por ter ~4x mais dados de treino que W2 e ~6x mais que W1. Ou seja, a agregacao por media
nao "discordou" muito do que uma otimizacao apenas em W3 já diria.

A diferenca real em relacao ao notebook 09 aparece num lugar inesperado: ao reajustar
(refit) exatamente os MESMOS hiperparametros que o notebook 09 escolheu (max_depth=6,
learning_rate=0.2, n_estimators=100, min_child_weight=10, subsample=1.0,
colsample_bytree=0.6), com a mesma seed e os mesmos dados de treino/validacao, o resultado
neste notebook foi diferente do original: AUC 0.6764 e lucro $ 190.805.872 aqui, contra
AUC 0.6744 e lucro $ 191.547.749,84 no notebook 09. Mesma configuracao, mesmos dados,
resultado numerico diferente. Isso e uma caracteristica conhecida do XGBoost: a construcao
de arvores por histograma com multiplas threads nao e bit-a-bit deterministica entre
execucoes separadas, mesmo com random_state fixo. Isso significa que parte da posicao
"#19 de 30" atribuida a combinacao do notebook 09 no ranking walk-forward, e parte da
comparacao pareada da Secao 6 contra XGB_tuned, carregam um ruido de reexecucao da ordem
de ~$700 mil - proximo da magnitude do proprio efeito que a Secao 6 tenta medir. **Isso
deveria ter sido verificado antes de reportar a comparacao XGB_walkforward vs XGB_tuned
como um resultado limpo - registrando aqui como limitacao, nao escondendo.**

**(b) A instabilidade de hiperparametros entre janelas e real, mas confundida com o
tamanho do treino, nao apenas com o tempo.**

Os hiperparametros nao sao estaveis: max_depth cresce de forma monotonica com o tamanho da
janela (W1=3, W2=4, W3=8), enquanto learning_rate cai de 0.20 (nb09, ajuste janela unica)
para 0.01-0.03 nas janelas walk-forward. Isso e evidencia de nao-estacionariedade no
sentido em que o notebook 08/09 levantaram a pergunta - nenhum unico hiperparametro serve
bem para todas as 3 janelas. Mas como as janelas sao expansivas (cada uma contem
integralmente a anterior), tamanho de treino e tempo estao confundidos por construcao: nao
da para saber, com este desenho, se a arvore precisa ficar mais profunda porque o processo
gerador mudou ao longo do tempo, ou simplesmente porque mais dados sustentam mais
complexidade sem overfitting - as duas explicacoes preveem exatamente o mesmo padrao
observado (profundidade crescente da janela 1 para a 3). Um desenho com janelas de
tamanho FIXO (nao expansivas) seria necessario para separar as duas hipoteses; este
notebook nao faz essa distincao.

**(c) Candidato final para o teste.**

Neste notebook, XGB_walkforward tem a maior AUC-ROC (0.6815) e o maior lucro na validacao
de 2014 ($ 191.757.692,42) entre todos os modelos comparados, e os ganhos vs M0b (+$
2.007.604,66, IC95% [$1.120.036, $2.889.184], nao cruza zero) e vs M1 (+$ 1.125.445,42,
IC95% [$278.778, $1.982.918], nao cruza zero, vence 99,4% das reamostras) sao robustos a
reamostragem. A comparacao vs XGB_tuned tambem nao cruza zero (IC95% [$19.539,
$1.869.180]), mas dado o ruido de reexecucao do XGBoost documentado no item (a), de
magnitude comparavel ao limite inferior desse intervalo, esse resultado especifico deveria
ser tratado com mais cautela do que os outros dois - a superioridade de XGB_walkforward
sobre M0b e M1 e mais solida do que a sua superioridade sobre XGB_tuned.

Com essa ressalva, XGB_walkforward segue como candidato mais forte para seguir ao teste:
tem o melhor desempenho na validacao de 2014 entre os modelos considerados, e sua
selecao por lucro medio em 3 janelas (em vez de uma unica) e, em principio, um criterio
mais conservador contra overfitting ao ano especifico de validacao - ainda que, como
discutido em (a), a maior parte do peso dessa media venha da propria janela 2014.

**O que NAO fica estabelecido aqui:**
- Que a escolha walk-forward supera a escolha janela-unica de forma inequivoca - o gap
  observado e da mesma ordem de grandeza do ruido de reexecucao do XGBoost.
- Que a instabilidade de hiperparametros entre janelas reflete no-estacionariedade
  temporal pura - o desenho expansivo confunde tempo com tamanho de treino.
- Nenhuma leitura sobre o conjunto de teste, que continua intocado.